# 77. 원판 추적과 코사인 상승량이 FP16에서도 유지될까?

Mac 베이스라인과 CutMix는 검증 영상 첫 장부터 각각 이전 궤적을 기억합니다. Mac 베이스라인의 출력이 비었을 때만 CutMix를 선택합니다. 코사인 모델은 무릎 통과와 접촉 두 장에서만 독립 추론합니다. 아래 셀은 Mac에서 채점이 끝난 결과를 엽니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wlcv.config import ROOT

folder = ROOT / "outputs/plate_aux_trt/20261126"
engines = ["pytorch", "onnx", "trt"]
summary = pd.read_csv(folder / "summary.csv").set_index("engine")
display(summary[["judged","tp","fn","fp","gap","longest","gained","lost","endpoints","measured","mae"]].round(5))
changes = pd.read_csv(folder / "changes.csv")
display(changes[["engine","video_id","frame_index","gained","lost","selected_model"]])

## 여섯 선수의 허브 상승량

오차는 예측한 상승량에서 수동 허브 상승량을 뺀 값입니다. 왼쪽은 선수별 측정 오차, 오른쪽은 FP16으로 바꾼 뒤 예측 상승량 자체가 달라진 폭입니다. 두 그래프의 세로축 눈금이 다릅니다.

In [ ]:
athletes = pd.concat([pd.read_csv(folder / f"{name}-athletes.csv").assign(engine=name) for name in engines])
errors = athletes.pivot(index="video_id",columns="engine",values="error_px")[engines]
rise = athletes.pivot(index="video_id",columns="engine",values="rise_px")
display(athletes[["engine","video_id","fn","fp","source","manual_px","rise_px","error_px"]].round(4))
fig, axes = plt.subplots(1,2,figsize=(12,4),layout="constrained")
errors.rename(index=lambda video: video[:3]).plot.bar(ax=axes[0],rot=0)
delta = (rise.trt - rise.pytorch).rename(index=lambda video: video[:3])
delta.plot.bar(ax=axes[1],color="#aa5d40",rot=0)
for ax, title in zip(axes,["Predicted minus manual rise","TensorRT FP16 minus PyTorch FP32 rise"]):
    ax.axhline(0,color="#444444",lw=0.8)
    ax.set(xlabel="Athlete",ylabel="Vertical displacement (px)",title=title)
fig.savefig(folder / "rise_comparison.png",dpi=140)
plt.show()

## 어떤 프레임을 놓쳤고 어느 시점이 코사인 검출을 통과했나

수동 박스가 있는 동작 프레임만 미탐·오탐을 채점합니다. 코사인이 한 시점이라도 보류된 선수는 두 시점 모두 기존 추적 중심을 사용합니다.

In [ ]:
for engine in engines:
    track = pd.read_csv(folder / f"{engine}-frames.csv")
    misses = track[track.in_motion & track.scored & ~track.hit]
    print(engine, list(zip(misses.video_id,misses.frame_index)))
    points = pd.read_csv(folder / f"{engine}-endpoints.csv")
    display(points[["video_id","frame_index","tracking_model","inter_model_iou","accepted","reason"]].round(3))

## 실제 프레임에서 박스 확인하기
a03 10번은 CutMix가 복구한 장면이고 b02 35번은 접촉 직전의 장면입니다. 초록 점선은 수동 박스, 주황색은 해당 런타임이 최종 선택한 박스입니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import DATASET, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.schedule_review import review_inputs
truth = review_inputs(DATASET / "data.yaml")[1].set_index(["video_id","frame_index"])
samples = [("a03_sn_f_01",10),("b02_sn_h_02",35)]
fig, axes = plt.subplots(2,3,figsize=(10,6),layout="constrained")
for key, axs in zip(samples,axes):
    video,index = key
    image = read_rgb(VIDEOS / f"{video}.mp4",index)
    gt = truth.loc[key]
    for engine, ax in zip(engines,axs):
        frame = pd.read_csv(folder / f"{engine}-frames.csv").set_index(["video_id","frame_index"]).loc[key]
        ax.imshow(image)
        for box,color,style in [(gt,"lime","--"),(frame,"darkorange","-")]:
            if pd.notna(box.x1):
                ax.add_patch(Rectangle((box.x1,box.y1),box.x2-box.x1,box.y2-box.y1,fill=False,edgecolor=color,ls=style,lw=2))
        w,h = gt.x2-gt.x1,gt.y2-gt.y1
        ax.set(xlim=(max(0,gt.x1-w/2),min(image.shape[1],gt.x2+w/2)),
               ylim=(min(image.shape[0],gt.y2+h/2),max(0,gt.y1-h/2)),
               title=f"{engine} | {video[:3]} #{index} | {frame.selected_model}")
        ax.axis("off")
fig.savefig(folder / "frames_local.png",dpi=130)
plt.show()